In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

MODELS_DIR = '/content/drive/MyDrive/FranchiseOps_AI/kaggle'
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Models will be saved to: {MODELS_DIR}")

!pip install -q scikit-learn pandas numpy joblib xgboost

Mounted at /content/drive
Models will be saved to: /content/drive/MyDrive/FranchiseOps_AI/kaggle


In [ ]:
import os, glob, pandas as pd, numpy as np, joblib
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, r2_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier, RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import LogisticRegression, Ridge, Lasso, ElasticNet
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB
import warnings
warnings.filterwarnings('ignore')

try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("✅ Kaggle credentials successfully loaded from Colab Secrets.")
except Exception as e:
    print("⚠️ Kaggle secrets not found in Colab userdata.")

def generate_synthetic_fallback(target_keyword, task_type='classification', N=5000):
    print(f"🤖 Generating Synthetic Data for '{target_keyword}'...")
    np.random.seed(42)
    X = pd.DataFrame({
        'synth_feature_1': np.random.normal(100, 15, N),
        'synth_feature_2': np.random.uniform(0, 1, N),
        'synth_feature_3': np.random.randint(1, 10, N),
        'synth_feature_4': np.random.poisson(5, N)
    })
    noise = np.random.normal(0, 0.1, N)
    signal = (X['synth_feature_1']/100) + X['synth_feature_2'] + (X['synth_feature_3']/10) + noise

    if task_type == 'classification':
        y = (signal > np.median(signal)).astype(int)
    else:
        y = signal * 1000

    X['TARGET_VAR'] = y
    return X

def process_kaggle_datasets(dataset_list, target_keyword, task_type='classification'):
    all_dfs = []

    for dset in dataset_list:
        print(f"\n--- Attempting {dset} ---")
        os.system(f"kaggle datasets download -d {dset} --unzip -q")

        csv_files = glob.glob("*.csv")
        if not csv_files:
            print(f"❌ No CSV found for {dset}.")
            continue

        found_target = False
        for csvf in csv_files:
            try:
                df = pd.read_csv(csvf, encoding='utf-8', on_bad_lines='skip')
            except Exception as e:
                try:
                    df = pd.read_csv(csvf, encoding='latin1', on_bad_lines='skip')
                except:
                    continue

            target_col = None
            for col in df.columns:
                if target_keyword.lower() in str(col).lower():
                    target_col = col
                    break

            if target_col:
                print(f"✅ Target '{target_col}' identified in {csvf}")
                # Rename target
                df.rename(columns={target_col: 'TARGET_VAR'}, inplace=True)
                # Drop massive text columns to save memory
                for col in df.columns:
                    if df[col].dtype == 'object' and df[col].nunique() > 1000:
                        df.drop(col, axis=1, inplace=True)

                # Label encode remaining objects
                for col in df.columns:
                    if df[col].dtype == 'object':
                        df[col] = LabelEncoder().fit_transform(df[col].astype(str))

                # Subsample if massive
                if len(df) > 5000:
                    df = df.sample(n=5000, random_state=42)

                all_dfs.append(df)
                found_target = True
                break

        if not found_target:
            print(f"❌ Target keyword '{target_keyword}' not found in any CSVs inside {dset}.")

        os.system("rm -f *.csv")

    # Always append synthetic
    synth_df = generate_synthetic_fallback(target_keyword, task_type)
    all_dfs.append(synth_df)

    print(f"\n🔗 Merging {len(all_dfs)} data sources into master dataset...")
    master_df = pd.concat(all_dfs, ignore_index=True)
    master_df.fillna(0, inplace=True) # Impute completely missing columns with 0

    print(f"Master Dataset Shape: {master_df.shape}")

    X = master_df.drop('TARGET_VAR', axis=1)
    y = master_df['TARGET_VAR']

    if task_type == 'classification':
        if len(y.unique()) > 10:
            y = (y > y.median()).astype(int)
        else:
            y = LabelEncoder().fit_transform(y)

    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    return Xtr, Xte, ytr, yte


✅ Kaggle credentials successfully loaded from Colab Secrets.


In [ ]:
try:
    datasets = ["manjeetsingh/retaildataset", "tanayatipre/store-sales-forecasting-dataset", "pereprosov/retail-store-performance"]
    Xtr, Xte, ytr, yte = process_kaggle_datasets(datasets, "Sales", "regression")

    if "regression" == "classification":
        models = {
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingClassifier(random_state=42),
            'Logistic Regression': LogisticRegression(max_iter=1000),
            'Neural Network': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeClassifier(max_depth=5),
            'SVM': SVC(probability=True),
            'KNN': KNeighborsClassifier(n_neighbors=5),
            'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42),
            'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
            'Naive Bayes': GaussianNB()
        }
    else:
        models = {
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingRegressor(random_state=42),
            'Ridge Regression': Ridge(),
            'Lasso Regression': Lasso(),
            'SVR': SVR(kernel='rbf'),
            'Neural Network': MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeRegressor(max_depth=5),
            'ElasticNet': ElasticNet(),
            'KNN': KNeighborsRegressor(n_neighbors=5),
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42)
        }

    best_score = -99
    best_pipe = None
    best_name = ""
    for name, m in models.items():
        try:
            pipe = Pipeline([('s', StandardScaler()), ('m', m)])
            pipe.fit(Xtr, ytr)
            score = accuracy_score(yte, pipe.predict(Xte)) if "regression" == "classification" else r2_score(yte, pipe.predict(Xte))
            if score > best_score:
                best_score = score
                best_pipe = pipe
                best_name = name
        except Exception as model_e:
            pass

    model_path = f"{MODELS_DIR}/agent1_franchise_model.joblib"
    joblib.dump(best_pipe, model_path)
    print(f"\n✅ Saved best model ({best_name}) to {model_path}")
except Exception as e:
    print(f"❌ Failed to process Agent 1: Outlet Performance: {e}")



--- Attempting manjeetsingh/retaildataset ---
✅ Target 'Weekly_Sales' identified in sales data-set.csv

--- Attempting tanayatipre/store-sales-forecasting-dataset ---
✅ Target 'Sales' identified in stores_sales_forecasting.csv

--- Attempting pereprosov/retail-store-performance ---
✅ Target 'MonthlySalesRevenue' identified in Store_CA.csv
🤖 Generating Synthetic Data for 'Sales'...

🔗 Merging 4 data sources into master dataset...
Master Dataset Shape: (13771, 39)

✅ Saved best model (Random Forest) to /content/drive/MyDrive/FranchiseOps_AI/kaggle/agent1_franchise_model.joblib


In [ ]:
try:
    datasets = ["jayjoshi37/inventory-demand-forecasting-and-stockout-risk", "ziya07/high-dimensional-supply-chain-inventory-dataset", "anirudhchauhan/retail-store-inventory-forecasting-dataset"]
    Xtr, Xte, ytr, yte = process_kaggle_datasets(datasets, "Stockout", "classification")

    if "classification" == "classification":
        models = {
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingClassifier(random_state=42),
            'Logistic Regression': LogisticRegression(max_iter=1000),
            'Neural Network': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeClassifier(max_depth=5),
            'SVM': SVC(probability=True),
            'KNN': KNeighborsClassifier(n_neighbors=5),
            'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42),
            'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
            'Naive Bayes': GaussianNB()
        }
    else:
        models = {
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingRegressor(random_state=42),
            'Ridge Regression': Ridge(),
            'Lasso Regression': Lasso(),
            'SVR': SVR(kernel='rbf'),
            'Neural Network': MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeRegressor(max_depth=5),
            'ElasticNet': ElasticNet(),
            'KNN': KNeighborsRegressor(n_neighbors=5),
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42)
        }

    best_score = -99
    best_pipe = None
    best_name = ""
    for name, m in models.items():
        try:
            pipe = Pipeline([('s', StandardScaler()), ('m', m)])
            pipe.fit(Xtr, ytr)
            score = accuracy_score(yte, pipe.predict(Xte)) if "classification" == "classification" else r2_score(yte, pipe.predict(Xte))
            if score > best_score:
                best_score = score
                best_pipe = pipe
                best_name = name
        except Exception as model_e:
            pass

    model_path = f"{MODELS_DIR}/agent2_franchise_model.joblib"
    joblib.dump(best_pipe, model_path)
    print(f"\n✅ Saved best model ({best_name}) to {model_path}")
except Exception as e:
    print(f"❌ Failed to process Agent 2: Inventory Optimization: {e}")



--- Attempting jayjoshi37/inventory-demand-forecasting-and-stockout-risk ---
✅ Target 'stockout_risk' identified in inventory_demand_stockout_risk.csv

--- Attempting ziya07/high-dimensional-supply-chain-inventory-dataset ---
✅ Target 'Stockout_Flag' identified in supply_chain_dataset1.csv

--- Attempting anirudhchauhan/retail-store-inventory-forecasting-dataset ---
❌ Target keyword 'Stockout' not found in any CSVs inside anirudhchauhan/retail-store-inventory-forecasting-dataset.
🤖 Generating Synthetic Data for 'Stockout'...

🔗 Merging 3 data sources into master dataset...
Master Dataset Shape: (12800, 26)

✅ Saved best model (Neural Network) to /content/drive/MyDrive/FranchiseOps_AI/kaggle/agent2_franchise_model.joblib


In [ ]:
try:
    datasets = ["ishadss/productivity-prediction-of-garment-employees", "mexwell/employee-performance-and-productivity-data", "ziya07/employee-performance-dataset"]
    Xtr, Xte, ytr, yte = process_kaggle_datasets(datasets, "productivity", "regression")

    if "regression" == "classification":
        models = {
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingClassifier(random_state=42),
            'Logistic Regression': LogisticRegression(max_iter=1000),
            'Neural Network': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeClassifier(max_depth=5),
            'SVM': SVC(probability=True),
            'KNN': KNeighborsClassifier(n_neighbors=5),
            'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42),
            'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
            'Naive Bayes': GaussianNB()
        }
    else:
        models = {
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingRegressor(random_state=42),
            'Ridge Regression': Ridge(),
            'Lasso Regression': Lasso(),
            'SVR': SVR(kernel='rbf'),
            'Neural Network': MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeRegressor(max_depth=5),
            'ElasticNet': ElasticNet(),
            'KNN': KNeighborsRegressor(n_neighbors=5),
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42)
        }

    best_score = -99
    best_pipe = None
    best_name = ""
    for name, m in models.items():
        try:
            pipe = Pipeline([('s', StandardScaler()), ('m', m)])
            pipe.fit(Xtr, ytr)
            score = accuracy_score(yte, pipe.predict(Xte)) if "regression" == "classification" else r2_score(yte, pipe.predict(Xte))
            if score > best_score:
                best_score = score
                best_pipe = pipe
                best_name = name
        except Exception as model_e:
            pass

    model_path = f"{MODELS_DIR}/agent3_franchise_model.joblib"
    joblib.dump(best_pipe, model_path)
    print(f"\n✅ Saved best model ({best_name}) to {model_path}")
except Exception as e:
    print(f"❌ Failed to process Agent 3: Staff Productivity: {e}")



--- Attempting ishadss/productivity-prediction-of-garment-employees ---
✅ Target 'targeted_productivity' identified in garments_worker_productivity.csv

--- Attempting mexwell/employee-performance-and-productivity-data ---
❌ Target keyword 'productivity' not found in any CSVs inside mexwell/employee-performance-and-productivity-data.

--- Attempting ziya07/employee-performance-dataset ---
❌ Target keyword 'productivity' not found in any CSVs inside ziya07/employee-performance-dataset.
🤖 Generating Synthetic Data for 'productivity'...

🔗 Merging 2 data sources into master dataset...
Master Dataset Shape: (6197, 19)

✅ Saved best model (Neural Network) to /content/drive/MyDrive/FranchiseOps_AI/kaggle/agent3_franchise_model.joblib


In [ ]:
try:
    datasets = ["rabieelkharoua/predict-conversion-in-digital-marketing-dataset", "jsonk11/social-media-advertising-dataset", "manishabhatt22/marketing-campaign-performance-dataset"]
    Xtr, Xte, ytr, yte = process_kaggle_datasets(datasets, "Conversion", "classification")

    if "classification" == "classification":
        models = {
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingClassifier(random_state=42),
            'Logistic Regression': LogisticRegression(max_iter=1000),
            'Neural Network': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeClassifier(max_depth=5),
            'SVM': SVC(probability=True),
            'KNN': KNeighborsClassifier(n_neighbors=5),
            'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42),
            'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
            'Naive Bayes': GaussianNB()
        }
    else:
        models = {
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingRegressor(random_state=42),
            'Ridge Regression': Ridge(),
            'Lasso Regression': Lasso(),
            'SVR': SVR(kernel='rbf'),
            'Neural Network': MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeRegressor(max_depth=5),
            'ElasticNet': ElasticNet(),
            'KNN': KNeighborsRegressor(n_neighbors=5),
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42)
        }

    best_score = -99
    best_pipe = None
    best_name = ""
    for name, m in models.items():
        try:
            pipe = Pipeline([('s', StandardScaler()), ('m', m)])
            pipe.fit(Xtr, ytr)
            score = accuracy_score(yte, pipe.predict(Xte)) if "classification" == "classification" else r2_score(yte, pipe.predict(Xte))
            if score > best_score:
                best_score = score
                best_pipe = pipe
                best_name = name
        except Exception as model_e:
            pass

    model_path = f"{MODELS_DIR}/agent4_marketing_model.joblib"
    joblib.dump(best_pipe, model_path)
    print(f"\n✅ Saved best model ({best_name}) to {model_path}")
except Exception as e:
    print(f"❌ Failed to process Agent 4: Marketing Intelligence: {e}")



--- Attempting rabieelkharoua/predict-conversion-in-digital-marketing-dataset ---
✅ Target 'ConversionRate' identified in digital_marketing_campaign_dataset.csv

--- Attempting jsonk11/social-media-advertising-dataset ---
✅ Target 'Conversion_Rate' identified in Social_Media_Advertising.csv

--- Attempting manishabhatt22/marketing-campaign-performance-dataset ---
✅ Target 'Conversion_Rate' identified in marketing_campaign_dataset.csv
🤖 Generating Synthetic Data for 'Conversion'...

🔗 Merging 4 data sources into master dataset...
Master Dataset Shape: (20000, 39)

✅ Saved best model (SVM) to /content/drive/MyDrive/FranchiseOps_AI/kaggle/agent4_marketing_model.joblib


In [ ]:
try:
    datasets = ["sid321axn/audit-data", "laraibnadeem2023/employee-policy-compliance-dataset", "ziya07/corporate-tax-risk-assessment-dataset"]
    Xtr, Xte, ytr, yte = process_kaggle_datasets(datasets, "Risk", "classification")

    if "classification" == "classification":
        models = {
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingClassifier(random_state=42),
            'Logistic Regression': LogisticRegression(max_iter=1000),
            'Neural Network': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeClassifier(max_depth=5),
            'SVM': SVC(probability=True),
            'KNN': KNeighborsClassifier(n_neighbors=5),
            'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42),
            'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
            'Naive Bayes': GaussianNB()
        }
    else:
        models = {
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingRegressor(random_state=42),
            'Ridge Regression': Ridge(),
            'Lasso Regression': Lasso(),
            'SVR': SVR(kernel='rbf'),
            'Neural Network': MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeRegressor(max_depth=5),
            'ElasticNet': ElasticNet(),
            'KNN': KNeighborsRegressor(n_neighbors=5),
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42)
        }

    best_score = -99
    best_pipe = None
    best_name = ""
    for name, m in models.items():
        try:
            pipe = Pipeline([('s', StandardScaler()), ('m', m)])
            pipe.fit(Xtr, ytr)
            score = accuracy_score(yte, pipe.predict(Xte)) if "classification" == "classification" else r2_score(yte, pipe.predict(Xte))
            if score > best_score:
                best_score = score
                best_pipe = pipe
                best_name = name
        except Exception as model_e:
            pass

    model_path = f"{MODELS_DIR}/agent5_audit_model.joblib"
    joblib.dump(best_pipe, model_path)
    print(f"\n✅ Saved best model ({best_name}) to {model_path}")
except Exception as e:
    print(f"❌ Failed to process Agent 5: Audit Engine: {e}")



--- Attempting sid321axn/audit-data ---
✅ Target 'Risk' identified in trial.csv

--- Attempting laraibnadeem2023/employee-policy-compliance-dataset ---
❌ Target keyword 'Risk' not found in any CSVs inside laraibnadeem2023/employee-policy-compliance-dataset.

--- Attempting ziya07/corporate-tax-risk-assessment-dataset ---
✅ Target 'Tax_Risk_Score' identified in corporate_tax_risk.csv
🤖 Generating Synthetic Data for 'Risk'...

🔗 Merging 3 data sources into master dataset...
Master Dataset Shape: (10776, 36)

✅ Saved best model (Gradient Boosting) to /content/drive/MyDrive/FranchiseOps_AI/kaggle/agent5_audit_model.joblib


In [ ]:
try:
    datasets = ["kundanbedmutha/customer-sentiment-dataset", "mansithummar67/171k-product-review-with-sentiment-dataset", "muhammadzamin1/product-reviews-dataset-for-sentiment-analysis"]
    Xtr, Xte, ytr, yte = process_kaggle_datasets(datasets, "Sentiment", "classification")

    if "classification" == "classification":
        models = {
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingClassifier(random_state=42),
            'Logistic Regression': LogisticRegression(max_iter=1000),
            'Neural Network': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeClassifier(max_depth=5),
            'SVM': SVC(probability=True),
            'KNN': KNeighborsClassifier(n_neighbors=5),
            'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42),
            'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
            'Naive Bayes': GaussianNB()
        }
    else:
        models = {
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingRegressor(random_state=42),
            'Ridge Regression': Ridge(),
            'Lasso Regression': Lasso(),
            'SVR': SVR(kernel='rbf'),
            'Neural Network': MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeRegressor(max_depth=5),
            'ElasticNet': ElasticNet(),
            'KNN': KNeighborsRegressor(n_neighbors=5),
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42)
        }

    best_score = -99
    best_pipe = None
    best_name = ""
    for name, m in models.items():
        try:
            pipe = Pipeline([('s', StandardScaler()), ('m', m)])
            pipe.fit(Xtr, ytr)
            score = accuracy_score(yte, pipe.predict(Xte)) if "classification" == "classification" else r2_score(yte, pipe.predict(Xte))
            if score > best_score:
                best_score = score
                best_pipe = pipe
                best_name = name
        except Exception as model_e:
            pass

    model_path = f"{MODELS_DIR}/agent6_sentiment_model.joblib"
    joblib.dump(best_pipe, model_path)
    print(f"\n✅ Saved best model ({best_name}) to {model_path}")
except Exception as e:
    print(f"❌ Failed to process Agent 6: Customer Sentiment: {e}")



--- Attempting kundanbedmutha/customer-sentiment-dataset ---
✅ Target 'sentiment' identified in Customer_Sentiment.csv

--- Attempting mansithummar67/171k-product-review-with-sentiment-dataset ---
✅ Target 'Sentiment' identified in Equal.csv

--- Attempting muhammadzamin1/product-reviews-dataset-for-sentiment-analysis ---
❌ Target keyword 'Sentiment' not found in any CSVs inside muhammadzamin1/product-reviews-dataset-for-sentiment-analysis.
🤖 Generating Synthetic Data for 'Sentiment'...

🔗 Merging 3 data sources into master dataset...
Master Dataset Shape: (15000, 20)

✅ Saved best model (Gradient Boosting) to /content/drive/MyDrive/FranchiseOps_AI/kaggle/agent6_sentiment_model.joblib


In [ ]:
try:
    datasets = ["san-francisco/sf-restaurant-scores-lives-standard", "sahirmaharajj/restaurant-inspection-results", "loulouashley/inspection-score-restaurant-inspection"]
    Xtr, Xte, ytr, yte = process_kaggle_datasets(datasets, "Score", "regression")

    if "regression" == "classification":
        models = {
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingClassifier(random_state=42),
            'Logistic Regression': LogisticRegression(max_iter=1000),
            'Neural Network': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeClassifier(max_depth=5),
            'SVM': SVC(probability=True),
            'KNN': KNeighborsClassifier(n_neighbors=5),
            'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42),
            'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
            'Naive Bayes': GaussianNB()
        }
    else:
        models = {
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingRegressor(random_state=42),
            'Ridge Regression': Ridge(),
            'Lasso Regression': Lasso(),
            'SVR': SVR(kernel='rbf'),
            'Neural Network': MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeRegressor(max_depth=5),
            'ElasticNet': ElasticNet(),
            'KNN': KNeighborsRegressor(n_neighbors=5),
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42)
        }

    best_score = -99
    best_pipe = None
    best_name = ""
    for name, m in models.items():
        try:
            pipe = Pipeline([('s', StandardScaler()), ('m', m)])
            pipe.fit(Xtr, ytr)
            score = accuracy_score(yte, pipe.predict(Xte)) if "regression" == "classification" else r2_score(yte, pipe.predict(Xte))
            if score > best_score:
                best_score = score
                best_pipe = pipe
                best_name = name
        except Exception as model_e:
            pass

    model_path = f"{MODELS_DIR}/agent7_safety_model.joblib"
    joblib.dump(best_pipe, model_path)
    print(f"\n✅ Saved best model ({best_name}) to {model_path}")
except Exception as e:
    print(f"❌ Failed to process Agent 7: Compliance & Safety: {e}")



--- Attempting san-francisco/sf-restaurant-scores-lives-standard ---
✅ Target 'inspection_score' identified in restaurant-scores-lives-standard.csv

--- Attempting sahirmaharajj/restaurant-inspection-results ---
❌ Target keyword 'Score' not found in any CSVs inside sahirmaharajj/restaurant-inspection-results.

--- Attempting loulouashley/inspection-score-restaurant-inspection ---
✅ Target 'inspection_score' identified in restaurant-and-food-inspections-1.csv
🤖 Generating Synthetic Data for 'Score'...

🔗 Merging 3 data sources into master dataset...
Master Dataset Shape: (15000, 24)

✅ Saved best model (Neural Network) to /content/drive/MyDrive/FranchiseOps_AI/kaggle/agent7_safety_model.joblib
